# Session 20 · Function calling
## Giving BankAssist hands — and catching it when it grabs the wrong thing

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why tools exist, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 11. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, honest limits, interview questions, troubleshooting, assignment | No |

Every section in Part B follows the same six steps:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

**Which model?** Your `.env` decides, as always. The batch default is `openai/gpt-oss-20b` on Groq. The offline option
is `llama3.2:3b` on Ollama (`BASE_URL=http://localhost:11434/v1`, `API_KEY=ollama`, `MODEL=llama3.2:3b`). Every cell
runs unchanged on either. The small model makes today's mistakes easier to see, which is why the instructor demos on it.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A customer asks BankAssist: *"What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"* In Session 19 BankAssist
could remember the whole conversation, but it could still only do one thing: write text. So it writes a number. The
number looks right. It has the right number of digits, a rupee sign, a confident sentence around it. But a language
model does not compute an EMI; it predicts the characters that usually follow a question like that. Sometimes the
prediction lands on the right answer. Often it lands near it. The exact answer is **₹21,696** a month, and a three-line
Python function gets it right every single time.

That gap — plausible versus exact — is the reason function calling exists. Instead of guessing, the model writes a
small structured request: *"please run `calculate_emi` with principal 2500000, rate 8.5, tenure 240."* Your code reads
that request, decides whether to trust it, runs the real function, and hands the result back. The model then writes
the sentence for the customer around a number it did not invent.

Here is the part most tutorials skip: **the model never runs anything.** It only asks. Everything that touches a
number, a database or the outside world is your code, and your code decides. That is good news, because it means you
are in control — and it is a responsibility, because a model that asks badly will happily ask for the wrong thing.
Today you will watch that happen on a real model, and fix it three different ways, measured.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Describe a function as a **tool schema** | A model that never calls your function, or calls it at the wrong time | The tool's job description |
| Run the **round trip** by hand | The most common bug in the topic: a tool result with no matching `tool_call_id` | Ask, run, hand back, answer |
| **Validate arguments in code** | Dividing by zero, or quoting a ₹0 EMI, because the model typed `"0"` | Check the form before you process it |
| **Cap the loop** at 3 steps | A conversation that calls tools forever and never answers | A loop with a safety brake |
| **Route first** | A small model reaching for a calculator to answer "what time do you open?" | Triage before treatment |
| **Score tool use on a golden set** | Choosing a fix (or a model) because it "seemed better" | Session 17's evidence, applied to tools |

### Where this is used in industry
- **Banking (BFS):** EMI calculators, eligibility checks, rate lookups, balance and transaction queries — every number a
  customer sees should come from the bank's own systems, not from a model's memory of what rates usually look like.
- **Insurance:** premium quotes and claim-status lookups are tool calls; the model explains, the system computes.
- **Healthcare:** appointment availability and drug-interaction checks come from a tool backed by a real database,
  never from what the model thinks is probably true.
- **Retail:** order status, stock checks, return eligibility — each one a tool with typed arguments.

### Why interviewers care
1. "Does the model execute the function?" (No. It returns a request; your code executes. Say this first.)
2. "Walk me through one tool call." (Send tools → model returns `tool_calls` → you run it → you append the assistant
   message and a `tool` message with the matching `tool_call_id` → second call → answer.)
3. "What goes wrong with tool calling, and how do you guard against it?" (Over-calling, invented or mistyped arguments,
   loops. Validate in code, cap the steps, route first, and measure.)
4. "How would you choose between two models for a tool-using assistant?" (Run the same golden set on both and compare
   tool selection, argument accuracy and answer accuracy — exactly Section 9.)

<details><summary><b>▸ Read more: "function calling" vs "tool use"</b></summary>

Same thing, two names. OpenAI introduced it as *function calling* in 2023 and later renamed the API field to `tools`;
Anthropic and most frameworks say *tool use*. The idea and the round trip are identical across providers, which is why
today's code runs unchanged on Groq and on Ollama.
</details>

## A2. Where this session fits in the course

```
 S14  System prompt as a contract ──────► BankAssist's standing rules
 S16  Structured output + a classifier ─► tool ARGUMENTS are just JSON; Fix C's router is a classifier
 S17  Golden set, scoring decided first ► Section 9 scores tool use the same way
 S18  Data is untrusted ───────────────► tool arguments and tool results are untrusted too
 S19  The growing messages list ───────► tool requests and results are appended to the same list
  │
 S20  Function calling ────────────────► TODAY: schema · round trip · validate · route · measure
  │
 S21-22  LangChain / LiteLLM ──────────► @tool and bind_tools wrap exactly today's raw protocol
 S23     FastAPI + SSE ────────────────► tool calls inside a served, streamed conversation
 Module 6  Agents ─────────────────────► today's round trip inside a while-loop, plus human approval
 Module 8  MCP ────────────────────────► tool schemas as a shared, standard protocol
 Module 12 Security ───────────────────► tool permissions enforced in code
```

**What you need from earlier sessions:** a working `.env` (Session 11), the course repository pulled, and Session
19's files — today imports `BANKASSIST_SYSTEM` from `chat_tasks.py` rather than copying it.

## A3. The words you need today

---
**1. Function calling (tool use)**
- **Definition:** instead of answering in prose, the model replies with a structured request: the name of a function and
  its arguments as JSON. Your code runs the function and sends the result back.
- **Analogy:** a senior banker who cannot operate any system tells the clerk "pull up the EMI for 25 lakh at 8.5% over
  20 years." The clerk runs the calculator and reads the number back. The banker never touches the keyboard.
- **Comes back in:** S21-22, Module 6 (an agent is this, repeated), Module 8.

---
**2. Tool schema**
- **Definition:** a description of one function in JSON Schema: `name`, `description`, and typed `parameters`.
- **The surprise:** the `description` is a prompt. It is the only thing the model reads to decide *when* to call the
  tool. Session 14's prompt skills apply directly.
- **Often confused with:** a contract the model must obey. `type` and `required` are requests, not rules — Section 4
  shows a model breaking both.
- **Comes back in:** Module 8 (MCP standardises exactly this).

---
**3. `tool_calls` and `tool_call_id`**
- **Definition:** `tool_calls` is the list of requests in the model's reply (each with an `id`, a `name` and an
  `arguments` string). When you send the result back, your `tool` message carries `tool_call_id` = that `id`.
- **Analogy:** a ticket number at a service counter — the answer has to carry the same number as the request.
- **Comes back in:** S21, Module 6.

---
**4. `tool_choice`**
- **Definition:** a setting on the call: `auto` (the model decides — today's default), `none`, `required`, or one
  named function.
- **Comes back in:** S21, Module 6.

---
**5. Over-calling**
- **Definition:** calling a tool the question did not need. Small models are known for it: give them a tool and they
  tend to use it regardless.
- **Comes back in:** Module 6 evaluations.

---
**6. Argument hallucination**
- **Definition:** invented or mistyped arguments — zeros because nothing fits, strings where numbers belong, a
  loan amount the customer never mentioned.
- **Analogy:** a form filled in by someone guessing, in handwriting the system can't read.
- **Comes back in:** Module 6, Module 12.

---
**7. Router**
- **Definition:** a cheap first call that decides which path a question takes — here, "needs a tool" or "doesn't".
- **Analogy:** triage at a hospital entrance: decide where you go before anyone starts treating you.
- **Comes back in:** Module 6 multi-agent systems, where a router picks which agent handles a request.

## A4. How today is built

```
            ┌──────────────────────────────────────────────────────────────────┐
            │                    tool_tasks.py  (the data)                       │
            │  TOOLS (3 schemas)   RATE_CARD   FOIR_LIMIT = 0.50                 │
            │  TOOLS_GUIDANCE (Fix A)   ROUTER_PROMPT (Fix C)                    │
            │  GOLDEN: 10 questions with known right answers   WAYS (4)          │
            └──────────────────────────────┬───────────────────────────────────┘
                                           │        chat_tasks.py (S19): BANKASSIST_SYSTEM
            ┌──────────────────────────────▼───────────────────────────────────┐
            │                    tool_utils.py  (the toolbox)                     │
            │  calculate_emi · get_rate · check_eligibility   ← plain Python      │
            │  execute_tool        → runs one tool, always returns JSON           │
            │  validate_args       → Fix B: types and ranges checked in code       │
            │  call_model          → one raw API call, normalised, cached          │
            │  run_with_tools      → the round trip, capped at 3 steps             │
            │  route               → Fix C: TOOL or NO_TOOL first                  │
            │  score_golden / compare_fixes → three scores, four ways              │
            └──────────────────────────────┬───────────────────────────────────┘
                                           │
                              YOUR notebook sections 1–11
                                           │
                         experiments/s20_tools.md  (committed, in YOUR work repo)
```

Section order: feel the problem with no tools → one round trip by hand → watch a model misuse a tool → three tools
and a capped loop → Fix A, Fix B, Fix C one at a time → score all four ways on the golden set → save → checkpoint.

**Raw API on purpose.** No LangChain today. Session 21 rebuilds this lab in a framework, and the point of doing it
by hand first is that you will know exactly what the framework is hiding.

---
# Part B · Hands-on

## Section 1 · Setup
**What and why.** Load your `.env`, build an OpenAI-compatible client (it talks to Groq or Ollama — same code), turn
on the answer cache, and import today's two files. Nothing here is new machinery.

**What the next cell does.** 1) Finds the repository root (Jupyter may start inside `module03/`) and puts `module02` and `module03` on the import path. 2) Reads `.env`. 3) Builds the
client and a `DiskCache`, so re-running a cell is free and gives the same answer. 4) Imports today's data and tools.
5) Prints which model and endpoint you are on.

In [ ]:
import sys, json, re
from pathlib import Path
# Jupyter starts the kernel in the notebook's folder (module03/), not the repo root - find the root either way
ROOT = Path.cwd() if (Path.cwd() / "module02").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "module02"))
sys.path.insert(0, str(ROOT / "module03"))

from dotenv import load_dotenv
load_dotenv(override=True)

import os
from openai import OpenAI
from reasoning_utils import DiskCache

client = OpenAI(api_key=os.environ["API_KEY"], base_url=os.environ["BASE_URL"])
MODEL = os.environ["MODEL"]
cache = DiskCache(Path.home() / ".genai_course" / "s20_cache.json")

from tool_tasks import BANKASSIST_SYSTEM, GOLDEN, RATE_CARD, TOOLS, TOOLS_GUIDANCE, ROUTER_PROMPT, WAYS
from tool_utils import (answer_question, assistant_message, calculate_emi, call_model, check_s20_work,
                        compare_fixes, execute_tool, fixes_table, parse_args, question_grid, route, run_with_tools,
                        save_tools_report, score_question, show_trace, tool_message, tools_markdown, validate_args)

print(f"model:    {MODEL}")
print(f"endpoint: {os.environ['BASE_URL']}" + ("   (local Ollama)" if "11434" in os.environ["BASE_URL"] else ""))
print(f"tools:    {[t['function']['name'] for t in TOOLS]}")
print(f"golden:   {len(GOLDEN)} questions, {sum(1 for g in GOLDEN if g['expect_tools'])} need a tool")

**What you should see.** Your model name, the endpoint, the three tool names, and `10 questions, 6 need a tool`.

**Check yourself.** A `404 model not found` later means `MODEL` doesn't exactly match a name your provider has. On
Ollama, run `ollama list` and copy the name exactly (for example `llama3.2:3b`).

## Section 2 · Predict first: an EMI with no tools
**What and why.** Before giving BankAssist any tools, ask it for an EMI the only way it can answer: in its head. Then
compute the exact answer in Python and compare. This is the whole case for tools, in two numbers.

**The idea.** A language model predicts the next characters. An EMI is a formula with a power of 240 in it. Predicting
digits is not computing them, so the model's number is usually *plausible* — and often not *exact*.

**Predict.** Before you run it: will the model's number match ₹21,696 exactly? Write your guess down.

**What the next cell does.** 1) Sends the question with BankAssist's system prompt and **no tools**. 2) Prints the
reply. 3) Computes the exact EMI with `calculate_emi`. 4) Pulls every number out of the reply so you can compare.

In [ ]:
question = "What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"
out = call_model(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                 {"role": "user", "content": question}], tools=None, cache=cache)
print("model, no tools:", out["content"])
print()
exact = calculate_emi(2_500_000, 8.5, 240)
print(f"exact, by formula: Rs {exact:,.2f}  ->  Rs {round(exact):,} a month")
numbers = [float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*\.?\d*", out["content"])]
print("numbers in the model's reply:", numbers)

**What you should see.** A confident, polite answer. Its number may be close to 21,696, may be far off, or the model
may decline because the fact sheet has no EMI table (a well-behaved answer, but still not the number). The formula
gives 21,695.58, which rounds to **₹21,696**.

**Check yourself.** If the model got it exactly right, ask it for 30 lakh over 15 years (exact: ₹29,542) and see whether
it stays right. *Common mistake:* concluding "the model can do maths" from one lucky answer. You will measure it
properly in Section 9.

## Section 3 · One tool, one round trip — by hand
**What and why.** Give BankAssist exactly one tool, `calculate_emi`, and walk the full round trip with every message
printed. You need to see the two new message shapes once, by hand, before any helper hides them.

**The idea.**
```
 ① you  ──► messages + tools ──► model
 ② model ──► finish_reason "tool_calls", name + arguments (a JSON STRING)
 ③ you  ──► parse, run calculate_emi(...)
 ④ you  ──► append: the assistant message WITH tool_calls, then a "tool" message with the SAME tool_call_id
 ⑤ model ──► second call, sees the result, writes the answer
```
The model is called **twice** for one answer. Between the two calls, the model isn't running at all — your code is.

**What the next cell does.** 1) First call with `tools=[calculate_emi only]`. 2) Prints `finish_reason` and the raw
`tool_calls`. 3) Parses the arguments string (`parse_args` is `json.loads` plus a safety net) and runs the real function. 4) Appends the assistant
message and the tool message. 5) Prints the whole messages list so you can see both shapes. 6) Second call: the answer.

In [ ]:
ONE_TOOL = [TOOLS[0]]   # calculate_emi only
messages = [{"role": "system", "content": BANKASSIST_SYSTEM}, {"role": "user", "content": question}]

first = call_model(client, MODEL, messages, tools=ONE_TOOL, cache=cache)
print("finish_reason:", first["finish_reason"])
print("tool_calls:   ", first["tool_calls"])

if not first["tool_calls"]:
    print("No tool call this time - the model answered in prose:", first["content"][:200])
    print("Read 'If yours is different' below, then re-run.")
else:
    call = first["tool_calls"][0]
    args = parse_args(call["arguments"])            # a STRING until you parse it (None if not valid JSON)
    result = execute_tool(call["name"], args or {})  # runs the real Python function
    print("ran:", call["name"], args, "->", result)

    messages.append(assistant_message(first))       # new shape 1: assistant turn WITH tool_calls
    messages.append(tool_message(call["id"], result))   # new shape 2: role "tool", same id
    print()
    print(json.dumps(messages[1:], indent=1, ensure_ascii=False))

    final = call_model(client, MODEL, messages, tools=ONE_TOOL, cache=cache)
    print()
    print("answer:", final["content"])

**What you should see.** `finish_reason: tool_calls`, one call to `calculate_emi` with arguments close to
`{"principal": 2500000, "annual_rate": 8.5, "tenure_months": 240}`, a result of `{"emi_rupees": 21696}`, the printed
list ending in an `assistant` message with `tool_calls` and a `tool` message with the same id, and a final answer
quoting ₹21,696.

**If yours is different.** If the first call has no `tool_calls`, the model answered in prose instead — try once more
(a cached answer won't change, so delete `~/.genai_course/s20_cache.json` first), and check your model supports tools.
On gpt-oss, `max_tokens` must stay generous: hidden reasoning can use the budget before the tool call is written.

**Check yourself.** Why must the assistant message with `tool_calls` go into the list *before* the tool message?
(Because the tool message answers it. A result with no request before it, or a mismatched id, is rejected by the API —
the single most common bug in this topic.)

## Section 4 · Watch it grab the wrong tool
**What and why.** Same single tool, a question it has nothing to do with: branch timings. In class, most people predict
"no tool call." Run it and see what your model actually does.

**The idea.** A tool's description says when to use it, and the schema says what types the arguments must be. Both
are *requests*. On the instructor's laptop, `llama3.2:3b` answered this question with:
```
finish_reason: tool_calls
tool_calls: [('calculate_emi', '{"annual_rate":"0","principal":"0","tenure_months":"0"}')]
```
Three failures in one call: **over-calling** (a tool for a question no tool can answer), **invented arguments**
(zeros, because nothing fits), and a **broken schema** (strings where it says number).

**What the next cell does.** 1) Asks the branch-timings question with only `calculate_emi` attached. 2) Prints what came
back. 3) For every tool call: shows what `validate_args` says about it, and what would happen if you ran it without
checking.

In [ ]:
timings = "What are your branch timings on Saturday?"
out = call_model(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                 {"role": "user", "content": timings}], tools=ONE_TOOL, cache=cache)
print("finish_reason:", out["finish_reason"])
print("tool_calls:   ", out["tool_calls"])
print("text:         ", out["content"][:200] or "(empty)")
for c in out["tool_calls"]:
    clean, problem = validate_args(c["name"], c["arguments"])
    print()
    print("validate_args says:", "OK" if problem is None else f"INVALID - {problem}")
    raw = parse_args(c["arguments"]) or {}
    print("run it unchecked:  ", execute_tool(c["name"], raw))

**What you should see.** On a small model, very likely a `calculate_emi` call with zeros or strings, `INVALID` from the
validator, and an error from running it unchecked (division by zero, or text where a number belongs). On a larger model
(gpt-oss-20b) you may see no tool call at all and a correct "10am to 1pm" answer — which is itself the point of
Section 9's model comparison.

**Check yourself.** If your code had trusted those arguments, what would the customer have seen? (A crash, or worse, a
cheerful "your EMI is ₹0 a month".) *Common mistake:* treating this as a bug in your code. It is model behaviour, and
it is the reason for Sections 6 to 8.

## Section 5 · Three tools, and a loop that stops at three steps
**What and why.** Real questions need more than one tool, sometimes one after another. Add `get_rate` (today's rate
card — live data instead of numbers baked into a prompt) and `check_eligibility` (Northwind's rule: all EMIs together
at most 50% of monthly income). Then let a helper run the round trip in a loop.

**The idea.** *"What would my EMI be on a 30 lakh home loan at your floating rate, over 15 years?"* needs two calls
in order: first the rate, then the EMI using that rate.
```
 step 1  model → get_rate("home_loan", "floating")   ← 8.5
 step 2  model → calculate_emi(3000000, 8.5, 180)    ← 29,542
 step 3  model → prose, no tool_calls → stop
```
`run_with_tools` loops — call, run, append, call again — until the model answers in prose **or three calls have
been made**. The cap is deliberate. A loop with no cap is an agent, and agents come in Module 6 with their own
safeguards.

**What the next cell does.** 1) Builds the messages for the two-step question. 2) Runs `run_with_tools` with all three
tools. 3) Prints a readable trace: every call, whether its arguments were valid, what came back, and the answer.

In [ ]:
two_step = "What would my EMI be on a 30 lakh home loan at your floating rate, over 15 years?"
res = run_with_tools(client, MODEL, [{"role": "system", "content": BANKASSIST_SYSTEM},
                                     {"role": "user", "content": two_step}], tools=TOOLS, cache=cache)
print(show_trace(res))
print()
print(f"model calls: {res['steps']}   tokens: {res['prompt_tokens'] + res['completion_tokens']:,}   "
      f"messages in the list now: {len(res['messages'])}")

**What you should see.** Ideally `get_rate` then `calculate_emi`, and an answer of about ₹29,542. A small model may
skip `get_rate` and guess the rate, call the tools in one go, or hit the 3-step cap. Read the trace — each of those is
a specific, nameable failure, not "it didn't work."

**Check yourself.** Count the messages in the list: system + user + (assistant + tool) per step + nothing for the final
answer (it's returned, not appended). Session 19's lesson is still here: every tool use adds two messages, so the list
grows faster than a plain chat.

## Section 6 · Fix A: tell it when NOT to call a tool
**What and why.** The cheapest possible fix: one paragraph added to the system prompt. No extra calls, a few dozen
extra tokens. Try it on the question that went wrong, and on one that should still use a tool — a fix that stops good
calls too is not a fix.

**The idea.** The guidance paragraph (printed below) names the three situations that need a tool, lists the things that
don't, and says "never invent an argument." On a large model this often stops over-calling outright. On a 3B model,
expect partial improvement: the pull toward "I have a tool, let me use it" is strong.

**What the next cell does.** 1) Prints `TOOLS_GUIDANCE`. 2) Answers the branch-timings question with Fix A. 3) Answers
the EMI question with Fix A, to check good calls still happen.

In [ ]:
print(TOOLS_GUIDANCE)
for q in (timings, question):
    print("=" * 90)
    print("Q:", q)
    print(show_trace(answer_question(client, MODEL, q, "prompt", cache=cache)))

**What you should see.** Maybe no tool call for branch timings now, and still a correct tool call for the EMI. On a
3B model the timings question may still trigger a tool — note it, that is a real finding.

**Check yourself.** Why can't a prompt alone guarantee anything? (It's a request to the model, the same as the schema's
types were in Section 4.) *Common mistake:* adding ever-longer guidance until one example passes. Measure on the whole
golden set instead (Section 9).

## Section 7 · Fix B: validate in code, and send the error back
**What and why.** Treat tool arguments exactly like a form filled in by a stranger — Session 18's rule. `validate_args`
checks types, ranges and required fields *before* the function runs. If anything is wrong, the function does not run;
the error goes back to the model as the tool's result, worded so it can recover.

**The idea.**
```
 model asks calculate_emi("0","0","0")
   → validate_args: "principal must be a positive number, got '0'"
   → tool result: "INVALID: ... If the customer did not ask for an EMI, a rate or an eligibility check,
                   answer WITHOUT calling a tool."
   → model, second try: "On Saturdays we're open 10am to 1pm."
```
This is **self-correction**, and it costs one extra round trip each time the validator fires. It doesn't stop the wrong
call from being *made* — it stops it from doing *damage*.

**What the next cell does.** 1) Shows `validate_args` on three hand-written bad calls, so you can see each rule. 2)
Answers the branch-timings question with Fix B and prints the trace.

In [ ]:
for name, raw in [("calculate_emi", '{"principal":"0","annual_rate":"0","tenure_months":"0"}'),
                  ("calculate_emi", '{"principal": 2500000, "annual_rate": 0.085, "tenure_months": 240}'),
                  ("get_rate", '{"product": "personal_loan", "rate_type": "floating"}')]:
    print(f"{name}({raw})\n   -> {validate_args(name, raw)[1]}")
print("=" * 90)
print(show_trace(answer_question(client, MODEL, timings, "validate", cache=cache)))

**What you should see.** Three specific error messages — a string where a number belongs, a fraction where a percent
belongs (`0.085` instead of `8.5`, a classic), and a product/rate pair that isn't on the card. Then, on a model that
over-calls, an `INVALID` step followed by a correct prose answer.

**Check yourself.** Why is the error message written *to the model* ("answer without calling a tool"), not to the
customer? (The customer never sees tool results; the model reads them and decides what to do next.)

## Section 8 · Fix C: route first
**What and why.** Session 16's classifier comes back in a new job. Before the main call, one small call decides:
`TOOL` or `NO_TOOL`. Only `TOOL` questions get the tools list attached. A model with no tools attached physically
cannot over-call.

**The idea.**
```
 question ──► router: "TOOL or NO_TOOL?" ──┬── TOOL    ──► BankAssist + tools
                                           └── NO_TOOL ──► BankAssist, no tools
```
**One detail that bites everyone:** `"NO_TOOL"` contains the letters `TOOL`. A check like `"TOOL" in reply` sends every
question to the tools path. `route()` checks how the reply *starts* instead.

**What the next cell does.** 1) Prints the router prompt. 2) Routes four questions — two that need a tool, two that
don't — and prints each decision. 3) Answers the branch-timings question with Fix C.

In [ ]:
print(ROUTER_PROMPT)
print()
for q in [question, two_step, timings, "Thanks, that's all I needed today!"]:
    needs, tokens = route(client, MODEL, q, cache=cache)
    print(f"{'TOOL   ' if needs else 'NO_TOOL'}  ({tokens:>5} tokens)  {q}")
print("=" * 90)
print(show_trace(answer_question(client, MODEL, timings, "router", cache=cache)))

**What you should see.** `TOOL` for the two EMI questions, `NO_TOOL` for timings and thanks, and a branch-timings answer
with no tool calls at all.

**Check yourself.** What's the new failure Fix C introduces? (A wrong route: an EMI question sent down the `NO_TOOL`
path gets no calculator, and BankAssist guesses — Section 2's problem, back again.) That's why the router is scored too.

## Section 9 · Score it: 10 golden questions, three scores, four ways
**What and why.** Every fix looked good on the one question we tried it on. That's anecdote. Session 17's discipline:
a fixed set of questions with known right answers, scores decided *before* running, and the same set run every way.

**The idea.** `GOLDEN` has 10 questions: 6 need a tool (one needs two steps, one is a "not eligible" case) and 4 need
none (timings, prepayment, credit score, a thank-you). Each answer gets three yes/no scores:

| Score | Means | Counted over |
|---|---|---|
| **Right tool (or none)** | exactly the expected tools were called — none, for a no-tool question | all 10 |
| **Args valid** | every call passed `validate_args`, and the main tool got the right values | the 6 tool questions |
| **Answer correct** | the final text contains the expected number or fact | all 10 |

Four ways: `none` (tools attached, no fix), `prompt` (Fix A), `validate` (Fix B), `router` (Fix C).

**What the next cell does.** 1) Runs `compare_fixes` — 10 questions × 4 ways, cached, so a re-run is free. On a laptop
the first run takes a few minutes. 2) Prints the summary table. 3) Prints a per-question grid so you can see *which*
questions each fix helped.

In [ ]:
results = compare_fixes(client, MODEL, cache=cache)
print(fixes_table(results))
print()
print(question_grid(results))

**What you should see.** Numbers that differ by model. On a small model, expect `none` to lose most of the no-tool
questions (over-calling), Fix B to lift "answer correct" by recovering from bad calls, and Fix C to fix the no-tool
questions outright — unless the router mis-routes one. On a larger model the gaps are smaller, because the baseline is
already better. Read the token column too: Fix B and Fix C buy their improvement with extra calls.

**Check yourself.** Pick one question that failed under every fix and read its row in `results["none"]["rows"]` —
`final_text` and `tools_called`. Is it a model failure, or is the grading too strict (for example "1 PM" vs "1pm")?
Keyword grading is a real limit of this lab, named in Part C.

## Section 10 · Save the report
**What and why.** `tools_markdown` turns Section 9 into one file: the model name, the four-way summary, and the
per-question grid. It goes into **your own** work repository, never the course repository. The model name matters:
these numbers mean nothing without it.

**What the next cell does.** 1) Builds the report text. 2) Prints it. The second cell saves it once you flip the flag.

In [ ]:
WORK_REPO = os.getenv("WORK_REPO") or r"C:\\Users\\you\\Documents\\genai-course-work"   # <-- YOUR folder
report_text = tools_markdown(results, MODEL)
print(report_text)

In [ ]:
SAVE_REPORT = False   # flip to True once you are happy with the numbers above

if SAVE_REPORT:
    path = save_tools_report(WORK_REPO, report_text)
    print(f"saved: {path}")

## Section 11 · Checkpoint
**What and why.** `check_s20_work` reads your saved report back and checks it offline: the model is named, all four
ways are present, and each was scored on the full golden set. It does **not** require any particular fix to win —
that depends on the model, and the report is evidence, not a race.

In [ ]:
from prompt_utils import results_table

print(results_table(check_s20_work(WORK_REPO)))

**What you should see.** Every line `PASS`. Then commit and push `experiments/s20_tools.md` in your `genai-course-work`
repository, and run `uv run pytest tests/test_s20.py -rs` from the course repository.

---
# Part C · Wrap-up

## C1. Summary
- **The model never runs anything.** It writes a structured request — a function name and JSON arguments — and your
  code decides whether to run it.
- **The round trip is two model calls with your code in the middle**: tools in, `tool_calls` out, run, append the
  assistant message and a `tool` message with the matching `tool_call_id`, call again. Both new messages join Session
  19's growing list.
- **A tool's description is a prompt**, and its schema is a request. Models can and do break both.
- **Small models over-call and invent arguments** — measured on a laptop today, not claimed on a slide.
- **Three fixes, three costs:** a prompt is free and partial; validation catches every bad argument but needs an extra
  round trip; a router stops over-calling outright but adds a call per question and can mis-route.
- **Choose by measuring** on a golden set, on the model you will actually deploy.

## C2. Honest limits
- **Ten questions is a smoke test, not a benchmark.** Hand-picked, small, one model at a time. Quote every score with
  the model name, and expect a different set of 10 to move the numbers.
- **Keyword grading is crude.** "Answer correct" looks for a number or phrase in the final text. A correct answer worded
  unusually can fail; a wrong answer that happens to contain the keyword can pass. Section 9's "check yourself" asks you
  to read failures, not just count them.
- **Every fix can fail.** A prompt is a request. Validation catches bad arguments, not wrong decisions (a valid call to
  the wrong tool still runs). The router is itself a model and can mis-route.
- **No write-actions today.** All three tools only calculate or read. The moment a tool can record, send or move money,
  validation is not enough: you need permissions enforced in code and a human approval step. That's Module 6 (human
  approval) and Module 12 (permissions), not today.
- **Tool results are untrusted too.** Today's tools return numbers our own code computed. A tool that returns text
  written by someone else — a complaint, an email body — re-opens Session 18's door: indirect injection through a tool
  result.

## C3. Interview questions, answered
1. **Does the model execute the function?** No. It returns a request (name + JSON arguments). The application executes
   it — or refuses to — and sends the result back for a second call.
2. **Walk me through one tool call.** Send messages and a `tools` list; the reply has `finish_reason == "tool_calls"`;
   parse the `arguments` string; validate; run; append the assistant message (with `tool_calls`) and a `tool` message
   with the same `tool_call_id`; call again; the model answers.
3. **What goes wrong, and how do you guard against it?** Over-calling, wrong or mistyped arguments, loops. Validate
   arguments in code and feed errors back; cap the number of steps; route first when tools are only sometimes needed;
   and measure tool selection and argument accuracy on a golden set.
4. **Why validate if the schema already declares types?** The schema is a request to the model, not something the
   model enforces. Section 4 showed strings where the schema said number.
5. **How would you choose a model for a tool-using assistant?** Run the same golden set on each candidate and compare
   tool-selection, argument and answer accuracy, plus cost — then decide, with the numbers.

## C4. Common mistakes
- **Forgetting to append the assistant message with `tool_calls`** before the `tool` message, or using the wrong
  `tool_call_id`. The API rejects the list.
- **Passing `arguments` straight into the function.** It's a string; parse it, then validate it.
- **No cap on the loop.** A model that keeps asking for tools will keep you paying for calls forever.
- **A tiny `max_tokens` on a reasoning model.** gpt-oss spends hidden tokens first; with too small a ceiling the tool
  call never gets written.
- **Checking `"TOOL" in reply` for a router.** `"NO_TOOL"` contains it.
- **Judging a fix on the one question that motivated it.** Run the golden set.

## C5. Troubleshooting
| Symptom | Likely cause | Fix |
|---|---|---|
| `404 model not found` | `MODEL` doesn't match an installed/available name | `ollama list` or `uv run python list_models.py`; copy the name exactly |
| `Connection refused` on `localhost:11434` | Ollama isn't running | Start the Ollama app, or `ollama serve` |
| No `tool_calls` ever, on any question | Model without tool support, or `max_tokens` too small on gpt-oss | Use `llama3.2:3b` or `gpt-oss-20b`; keep `max_tokens` at 1500 |
| The JSON of a tool call appears inside `content` instead of `tool_calls` | Older Ollama, or a model that writes tool calls as text | Update Ollama; re-pull the model |
| API error about `tool_call_id` | Tool message without the matching assistant message, or wrong id | Append `assistant_message(out)` first; use `call["id"]` |
| Same answer every time even after changing code | The cache returned a stored answer | Delete `~/.genai_course/s20_cache.json` |
| Section 9 is slow | 40+ calls on a laptop CPU | Normal on first run; re-runs are cached |

## C6. Assignment
**Stack the fixes.** Fix B and Fix C are not either/or. Add a fifth way, `"router+validate"`: route first, and on the
`TOOL` path also validate arguments. (Hint: in `answer_question`, the router branch calls `run_with_tools` with
`validate=False` — copy it and pass `True`.) Run the golden set five ways and report: does stacking beat both on "answer
correct"? What does it cost in tokens? Write two sentences on which way you'd ship for BankAssist **on your model**,
and why — with the numbers.